In [2]:
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime

# ==============================
# 1. LOAD DATA
# ==============================
file_path = r"C:\Users\Harshitha.M\Downloads\Play Store Data (1).csv"

df = pd.read_csv(file_path)

# ==============================
# 2. CLEAN DATA
# ==============================

df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

# Convert Size to MB
def convert_size(x):
    try:
        x = str(x).strip()

        if x.endswith("M"):
            return float(x[:-1])
        elif x.endswith("k"):
            return float(x[:-1]) / 1024
        else:
            return None
    except:
        return None

df["Size_MB"] = df["Size"].apply(convert_size)

# Convert Installs to numbers
df["Installs_Num"] = (
    df["Installs"]
    .astype(str)
    .str.replace("+", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("Free", "0", regex=False)
)

df["Installs_Num"] = pd.to_numeric(
    df["Installs_Num"], errors="coerce"
)

# Convert Last Updated to date
df["Last Updated"] = pd.to_datetime(
    df["Last Updated"],
    errors="coerce"
)

# ==============================
# 3. FILTER DATA
# Rating >= 4.0
# Size < 10 MB
# Last update in January
# ==============================

filtered_df = df[
    (df["Rating"] >= 4.0) &
    (df["Size_MB"] < 10) &
    (df["Last Updated"].dt.month == 1)
].copy()

# ==============================
# 4. TOP 10 CATEGORIES
# BY NUMBER OF INSTALLS
# ==============================

top_categories = (
    filtered_df
    .groupby("Category")["Installs_Num"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

top_df = filtered_df[
    filtered_df["Category"].isin(top_categories)
]

# ==============================
# 5. CALCULATE VALUES
# ==============================

category_data = (
    top_df
    .groupby("Category")
    .agg(
        Average_Rating=("Rating", "mean"),
        Total_Reviews=("Reviews", "sum")
    )
    .reindex(top_categories)
    .reset_index()
)

# ==============================
# 6. CURRENT TIME
# ==============================

current_hour = datetime.now().hour

# Graph only from 3 PM to 5 PM
if 15 <= current_hour < 17:

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=category_data["Category"],
            y=category_data["Average_Rating"],
            name="Average Rating"
        )
    )

    fig.add_trace(
        go.Bar(
            x=category_data["Category"],
            y=category_data["Total_Reviews"],
            name="Total Reviews"
        )
    )

    fig.update_layout(
        title="Top 10 App Categories - Average Rating vs Total Reviews",
        xaxis_title="App Category",
        yaxis_title="Value",
        barmode="group",
        template="plotly_white",
        width=1100,
        height=600
    )

    fig.show()

else:
    print("Graph is available only between 3:00 PM and 5:00 PM IST.")

Graph is available only between 3:00 PM and 5:00 PM IST.
